In [ ]:
import pandas as pd
from io import BytesIO
from minio import Minio

In [ ]:
MINIO_ENDPOINT = "localhost:9000"
MINIO_BUCKET = "cgmacros"
# CGMacros-004 (Somente fotos)
PATIENTS = ["CGMacros-012", "CGMacros-039"]

In [ ]:
client = Minio(
    MINIO_ENDPOINT,
    access_key="minioadmin",
    secret_key="minioadmin",
    secure=False,
)

In [ ]:
base_path = 'bronze/CGMacros/'

In [ ]:
for patient_id in PATIENTS:
    object_name = f"{base_path}{patient_id}/{patient_id}.csv"

    try:
        response = client.get_object(MINIO_BUCKET, object_name)

        df = pd.read_csv(BytesIO(response.read()))

        print(f"--- Colunas do paciente: {patient_id} ---")
        print(df.columns.tolist())
        print("\n")

    except Exception as err:
        print(f"Erro ao acessar {object_name}: {err}")
    finally:
        if 'response' in locals():
            response.close()
            response.release_conn()

In [ ]:
dtypes = {
    'Libre GL':           'float64',
    'Dexcom GL':          'float64',
    'HR':                 'float64',
    'Calories (Activity)':'float64',
    'METs':               'float64',
    'Meal Type':          'string',
    'Calories':           'float64',
    'Carbs':              'float64',
    'Protein':            'float64',
    'Fat':                'float64',
    'Fiber':              'float64',
    'Amount Consumed':    'float64',
    'Image path':         'string' # descartar na silver n utilizado
}

In [ ]:
for patient_id in PATIENTS:
    source_path = f"bronze/CGMacros/{patient_id}/{patient_id}.csv"
    target_path = f"silver/CGMacros/{patient_id}/{patient_id}.parquet"

    try:
        response = client.get_object(MINIO_BUCKET, source_path)
        df = pd.read_csv(BytesIO(response.read()), dtype=dtypes, parse_dates=['Timestamp'])

        parquet_buffer = BytesIO()
        df.to_parquet(parquet_buffer, engine='pyarrow', index=False)
        parquet_buffer.seek(0)

        client.put_object(
            MINIO_BUCKET,
            target_path,
            data=parquet_buffer,
            length=parquet_buffer.getbuffer().nbytes,
            content_type='application/x-parquet'
        )

        print(f"Sucesso: {patient_id} convertido para {target_path}")

    except Exception as err:
        print(f"Erro ao processar {patient_id}: {err}")
    finally:
        if 'response' in locals():
            response.close()
            response.release_conn()